In [1]:
import pandas as pd

import spacy
import re
import warnings

warnings.filterwarnings("ignore")



In [2]:
df = pd.read_csv("../data/raw/fintech_reviews_clean.csv")

df.head()

,review,rating,date,bank,source
0,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,5,2026-05-14,CBE,Google Play
1,incredible,5,2026-05-14,CBE,Google Play
2,best app for financial sector,5,2026-05-13,CBE,Google Play
3,it's a good application,5,2026-05-13,CBE,Google Play
4,thank you cbe,5,2026-05-13,CBE,Google Play


In [4]:
df["review_id"] = range(1, len(df) + 1)

In [5]:
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    doc = nlp(text)

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop and not token.is_punct
    ]

    return " ".join(tokens)

In [6]:
df["clean_review"] = df["review"].apply(preprocess_text)

In [7]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [8]:
def analyze_sentiment(text):

    result = classifier(text[:512])[0]

    label = result["label"]
    score = result["score"]

    return pd.Series([label, score])

In [9]:
df[["sentiment_label", "sentiment_score"]] = (
    df["review"]
    .apply(analyze_sentiment)
)

In [10]:
def map_sentiment(label, score):

    if score < 0.60:
        return "NEUTRAL"

    return label

In [11]:
df.groupby("bank")["sentiment_score"].mean()

bank
BOA       0.963442
CBE       0.966556
Dashen    0.977409
Name: sentiment_score, dtype: float64

In [12]:
df.groupby("rating")["sentiment_score"].mean()

rating
1    0.978893
2    0.968515
3    0.968035
4    0.961602
5    0.963141
Name: sentiment_score, dtype: float64

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    max_features=50
)

X = vectorizer.fit_transform(df["clean_review"])

keywords = vectorizer.get_feature_names_out()

print(keywords)

['account' 'add' 'amazing' 'app' 'application' 'bad' 'balance' 'bank'
 'banking' 'banking app' 'cbe' 'customer' 'dashen' 'developer' 'easy'
 'easy use' 'ethiopia' 'experience' 'fast' 'feature' 'fix' 'good'
 'good app' 'great' 'issue' 'like' 'love' 'mobile' 'mobile banking'
 'money' 'need' 'new' 'nice' 'open' 'option' 'phone' 'problem' 'send'
 'service' 'simple' 'thank' 'time' 'transaction' 'transfer' 'try' 'update'
 'use' 'user' 'version' 'work']


In [15]:
def identify_theme(text):

    text = text.lower()

    if "login" in text or "otp" in text:
        return "Account Access Issues"

    elif "slow" in text or "transfer" in text:
        return "Transaction Performance"

    elif "ui" in text or "design" in text:
        return "UI & Design"

    else:
        return "Other"

In [16]:
df["identified_theme"] = df["review"].apply(identify_theme)

In [17]:
final_df = df[
    [
        "review_id",
        "review",
        "sentiment_label",
        "sentiment_score",
        "identified_theme"
    ]
]

In [18]:
final_df.to_csv(
    "../data/raw/task2_sentiment_analysis.csv",
    index=False
)